# سبق 13 - ایجنٹ میموری


## سیٹ اپ

یہ نوٹ بک دکھاتی ہے کہ کس طرح **Microsoft Agent Framework** (MAF) کا استعمال کرتے ہوئے ایک ٹریول بکنگ ایجنٹ بنایا جائے جس میں **مستقل یادداشت** ہو۔

آپ سیکھیں گے کہ ایجنٹ کی مختلف قسم کی یادداشتیں — ورکنگ، قلیل مدتی، اور طویل مدتی — اس بات کو کیسے متاثر کرتی ہیں کہ ایک ایجنٹ بات چیت کے دوران معلومات کو کس طرح محفوظ اور استعمال کرتا ہے۔

**ضروریات:**
- ایک Azure AI Foundry پروجیکٹ جس میں ایک چیٹ ماڈل تعینات کیا گیا ہو (جیسے `gpt-4o-mini`)۔
- Azure CLI میں لاگ ان۔ اپنے ٹرمینل میں `az login` چلائیں۔
- `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Azure AI Foundry پروجیکٹ کا اینڈ پوائنٹ۔
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات کردہ ماڈل کا نام۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ایجنٹ میموری کی اقسام

اے آئی ایجنٹس مختلف اقسام کی میموری استعمال کر سکتے ہیں، ہر ایک کا ایک مخصوص مقصد ہوتا ہے:

### ورکنگ میموری
بات چیت کا سلسلہ خود — ایک ہی سیشن میں تبادلہ کیے گئے پیغامات۔ ایجنٹ اسی سلسلے میں پہلے کے پیغامات کا حوالہ دے سکتا ہے تاکہ ہم آہنگی قائم رہے۔ MAF میں یہ **`agent.create_session()`** کے ذریعے بنائی جاتی ہے، جو ایک `AgentSession` واپس کرتا ہے۔

### قلیل مدتی میموری
وہ معلومات جو کام یا سیشن کے دوران برقرار رہتی ہے لیکن مستقل طور پر محفوظ نہیں کی جاتی۔ مثال کے طور پر، ایجنٹ ایک کثیر مرحلوں والی منصوبہ بندی کی بات چیت کے دوران حقائق جمع کر سکتا ہے اور انہیں حتمی منصوبہ تیار کرنے کے لیے استعمال کر سکتا ہے۔

### طویل مدتی میموری
وہ ترجیحات اور حقائق جو **سیشنز کے درمیان** برقرار رہتے ہیں۔ ایک واپس آنے والے صارف کو اپنی غذائی پابندیاں یا سفر کے انداز بار بار دہرانے کی ضرورت نہیں ہونی چاہیے۔ طویل مدتی میموری عام طور پر کسی خارجی ذخیرے — ڈیٹا بیس، فائل، یا ویکٹر انڈیکس — کی مدد سے ہوتی ہے اور ایجنٹ کو ٹولز کے ذریعے فراہم کی جاتی ہے۔


## ورکنگ میموری ود سیشنز

میموری کی سب سے آسان شکل گفتگو کا سیشن ہے۔ جب آپ ایک ہی سیشن (جو `agent.create_session()` کے ذریعے بنایا گیا ہو) کو مسلسل `agent.run()` کالز میں بھیجتے ہیں، تو ایجنٹ اس گفتگو کی پوری تاریخ دیکھ سکتا ہے اور پہلے کے تفصیلات یاد رکھ سکتا ہے۔

آئیے ایک ٹریول ایجنٹ بنائیں اور ورکنگ میموری کا مظاہرہ کریں۔


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ایجنٹ نے بجٹ کو صحیح طریقے سے یاد رکھا کیونکہ دونوں پیغامات ایک ہی سیشن شیئر کرتے ہیں۔ یہ **کام کرنے کی یادداشت** ہے — یہ صرف سیشن کی مدت کے لیے موجود ہوتی ہے۔

### نۓ تھریڈ کے ساتھ کیا ہوتا ہے؟

اگر ہم **نیا** سیشن بنائیں، تو ایجنٹ کو پچھلی گفتگو کی کوئی یاد نہیں ہوتی:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## طویل مدتی یادداشت کا نمونہ

صارف کی ترجیحات کو **سیشنز کے دوران** یاد رکھنے کے لیے، ہمیں ایک مستقل ذخیرہ چاہیے جو گفتگو کے دھاگے کے باہر موجود ہو۔ ایجنٹ اس ذخیرے تک پہنچنے کے لیے **ٹولز** استعمال کرتا ہے — وہ فعل جو معلومات محفوظ کرنے اور بازیافت کرنے کے لیے کال کر سکتا ہے۔

نیچے ہم ایک سادہ ان میموری ترجیحی ذخیرہ نافذ کرتے ہیں (پیداواری ماحول میں آپ اسے ڈیٹا بیس یا ویکٹر انڈیکس کے ذریعے سپورٹ کریں گے) اور اسے ایسے ٹولز کی صورت میں پیش کرتے ہیں جنہیں ایجنٹ استعمال کر سکتا ہے۔

### فن تعمیر
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### منظر نامہ 1 — پہلی بار استعمال کرنے والا سالگرہ کی سیر کا سفر بک کرتا ہے

سارہ پہلی بار وزٹ کر رہی ہے۔ ایجنٹ کو چاہیے کہ وہ اس کی ترجیحات کو آلات کے ذریعے محفوظ کرے اور انہیں ہوٹلوں کی سفارش کے لیے استعمال کرے۔


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### منظر نامہ 2 — سارہ ہفتوں بعد واپس آتی ہے

سارہ ایک **نئی تھریڈ** شروع کرتی ہے (نئی سیشن کی سیمولیشن کرتے ہوئے)۔ ورکنگ میموری خالی ہے، لیکن طویل مدتی ترجیح کا ذخیرہ ابھی بھی اس کی معلومات رکھتا ہے۔ ایجنٹ کو یہ معلومات حاصل کر کے سفارشات کو ذاتی بنانے کے لیے استعمال کرنا چاہیے۔


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصہ

اس سبق میں آپ نے ایجنٹ کی تین قسم کی میموری سیکھی اور انہیں Microsoft Agent Framework کے ساتھ کیسے نافذ کیا جائے:

| میموری کی قسم | MAF کا طریقہ کار | زندگی کی مدت |
|---|---|---|
| **ورکنگ** | `agent.create_session()` | ایک گفتگو |
| **مختصر مدتی** | ایک تھریڈ کے اندر جمع کردہ سیاق و سباق | ایک کام / سیشن |
| **طویل مدتی** | بیرونی اسٹور جس تک `@tool` فنکشنز کے ذریعہ رسائی حاصل کی جاتی ہے | سیشنز کے درمیان |

### اہم نکات
1. **`agent.create_session()`** ورکنگ میموری فراہم کرتا ہے — ایجنٹ ایک سیشن کے اندر پوری گفتگو کی تاریخ دیکھتا ہے۔
2. **نئے سیشنز سیاق و سباق کھو دیتے ہیں** — بغیر طویل مدتی میموری کے ایجنٹ ماضی کی گفتگو یاد نہیں کر سکتا۔
3. **`@tool` فنکشنز** خلا کو پُر کرتے ہیں — یہ ایجنٹ کو مستقل اسٹور سے معلومات محفوظ کرنے اور بازیافت کرنے دیتے ہیں۔
4. **شخصی نوعیت وقت کے ساتھ بہتر ہوتی ہے** — جتنا زیادہ ترجیحات محفوظ ہوں، ایجنٹ کی سفارشات اتنی ہی بہتر ہوں گی۔

### حقیقی دنیا کی درخواستیں
- **کسٹمر سروس**: صارف کی تاریخ اور ترجیحات کو یاد رکھنا
- **ذاتی معاونین**: دنوں یا ہفتوں کے دوران سیاق و سباق برقرار رکھنا
- **صحت کی دیکھ بھال**: مریض کی معلومات اور ترجیحات کا ٹریک رکھنا
- **ای-کامرس**: تاریخ کی بنیاد پر ذاتی نوعیت کی خریداری

### اگلے مراحل
- ان-میموری لغت کو ڈیٹا بیس یا ویکٹر اسٹور (جیسے Azure AI Search) سے تبدیل کریں
- وقت کی حساس معلومات کے لئے میموری کے ختم ہونے کا اضافہ کریں
- مشترکہ میموری کے ساتھ کثیر ایجنٹ نظام بنائیں
- علم گراف پر مبنی میموری کے لیے Cognee نوٹ بک دریافت کریں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستخطی اعلامیہ**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم یہ ذہن میں رکھیں کہ خودکار ترجموں میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں ہی معتبر ماخذ سمجھا جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ ہم اس ترجمے کے استعمال سے ہونے والی کسی بھی غلط فہمی یا تشریح کی ذمہ داری قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
